[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/097.%20The%20Last-Mile%20Delivery%20%26%20Micro-Fulfillment%20Problem/P97-Tier-5_executed.ipynb)

---


# 97. The Last-Mile Delivery and Micro-Fulfillment Problem
## Tier 5: An Integrated Digital Twin Prototype

### Purpose and workflow

This notebook builds a small simulation of a last-mile delivery system. It represents delivery requests, vehicles, traffic sensors, and weather stations; generates synthetic system data; estimates near-term demand; detects selected disruptions; and evaluates a fixed number of randomly generated response scenarios. It then reports operational indicators and visualizes the simulated system.

The solution method is a simulation-and-scenario-enumeration workflow. For each detected disruption, the code generates 1,200 candidate response scenarios, assigns each a score, and selects the candidate with the highest score. The candidates are random samples from simple ranges, not operational routes or fully specified interventions. This is a demonstration of scenario analysis, not a validated production optimizer.

### Inputs and outputs

- **Inputs:** synthetic city coordinates, a mixed vehicle fleet, delivery requests, traffic conditions, weather conditions, and scenario-model parameters defined in the code.
- **Outputs:** demand indices relative to a baseline, detected disruption records, system KPIs, the highest-scoring sampled scenario for each disruption, estimated delay and gross avoided-delay costs, and visual summaries.
- **Execution:** run cells from top to bottom. Random draws are used, and no random seed is set, so numerical results can differ between runs.

### Scope and modeling assumptions

The city is represented by a rectangular coordinate system measured in kilometres. Locations are sampled uniformly, and the nearest micro-fulfillment center is selected using straight-line Euclidean distance. Traffic and weather records are synthetic rather than live data. The demand model is a rule-based index, not a statistically fitted forecast. The scenario model does not construct routes, enforce vehicle capacity, or simulate individual deliveries. Its cost calculation estimates gross avoided delay penalties and does not subtract the scenario's modeled cost increase.

A production digital twin may connect to live systems, maintain two-way synchronization, validate forecasts against observed outcomes, and support operational decisions. Those capabilities are not implemented or evaluated in this notebook.

### Scenario-selection model

For a detected disruption, let $S = \{1, 2, \ldots, 1200\}$ be the set of sampled candidate scenarios. Candidate $i$ has a delay-reduction fraction $r_i$ and a cost-increase fraction $c_i$. The code scores each candidate and selects one with the largest score:

$$\text{score}_i = r_i - 0.5c_i, \qquad i \in S,$$

$$i^* \in \underset{i \in S}{\operatorname{argmax}}\; \text{score}_i.$$

Here, $r_i$ and $c_i$ are dimensionless fractions sampled from disruption-specific ranges, and $0.5$ is the scenario model's trade-off weight. The code implements this procedure by creating a list of candidate dictionaries and calling `max` with `overall_score` as the key. If candidates tie for the highest score, `max` returns the first one encountered.

The selected reduction is converted into estimated minutes saved using a disruption-specific calculation. Gross avoided-delay cost is then calculated as

$$\text{gross avoided-delay cost} = \text{delay savings in minutes} \times \text{assumed cost per delay minute}.$$

The assumed cost per delay minute is specified in the scenario-analysis code. Because the scenario cost increase affects the selection score but is not subtracted from gross avoided-delay cost, that reported dollar value is not a net benefit.

### What to examine

Compare the detected disruptions with the system KPIs, inspect how the selected sampled scenario depends on the scoring rule, and distinguish forecast variability from forecast accuracy. The notebook has no observed future demand against which accuracy could be measured. Likewise, elapsed runtime and scenario outputs describe the current execution only; they are not evidence of a target runtime or real-world savings.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import random
import time
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Represent the system state

A digital twin needs a structured representation of the entities whose state it tracks. The data classes below make the fields of a delivery request, vehicle, traffic observation, and weather observation explicit. Coordinates are stored as $(x,y)$ pairs in kilometres; package size and vehicle capacity are measured in cubic metres; vehicle speed is measured in kilometres per hour.

The `DigitalTwin` object stores collections of these entities together with system-level metrics. Its methods generate synthetic inputs, update selected observations, calculate a simple demand index, detect disruptions, evaluate candidate scenarios, and summarize KPIs.

In [ ]:
@dataclass
class DeliveryRequest:
    """A delivery request from a micro-fulfillment center to a customer."""

    id: int
    customer_id: int
    origin: Tuple[float, float]
    destination: Tuple[float, float]
    priority: str
    time_window: Tuple[datetime, datetime]
    package_size: float
    status: str
    created_at: datetime
    assigned_vehicle: Optional[int] = None
    estimated_delivery: Optional[datetime] = None


@dataclass
class Vehicle:
    """A delivery vehicle and its current operating state."""

    id: int
    type: str
    location: Tuple[float, float]
    capacity: float
    speed: float
    battery_level: float
    status: str
    current_route: List[int] = field(default_factory=list)
    next_destination: Optional[Tuple[float, float]] = None


@dataclass
class TrafficSensor:
    """A synthetic traffic observation at one location and time."""

    id: int
    location: Tuple[float, float]
    congestion_level: float
    average_speed: float
    incident_detected: bool
    timestamp: datetime


@dataclass
class WeatherStation:
    """A synthetic weather observation at one location and time."""

    id: int
    location: Tuple[float, float]
    temperature: float
    precipitation: float
    wind_speed: float
    visibility: float
    conditions: str
    timestamp: datetime


class DigitalTwin:
    """A simulation state for an integrated last-mile delivery system."""

    def __init__(self, city_size: Tuple[float, float] = (100, 100)):
        self.city_size = city_size
        self.current_time = datetime.now()

        self.delivery_requests: List[DeliveryRequest] = []
        self.vehicles: List[Vehicle] = []
        self.traffic_sensors: List[TrafficSensor] = []
        self.weather_stations: List[WeatherStation] = []

        self.kpi_history = []
        self.predictions = {}
        self.scenario_results = {}

        self.total_deliveries = 0
        self.on_time_deliveries = 0
        self.failed_deliveries = 0
        self.total_distance = 0.0

    def initialize_system(self) -> None:
        """Create fulfillment centers, vehicles, sensors, and weather stations."""
        print("Initializing the digital twin system...")

        self.mfc_locations = [
            (25, 25),
            (75, 25),
            (25, 75),
            (75, 75),
            (50, 50),
        ]

        vehicle_configs = [
            ("van", 12, 50, 8),
            ("bike", 2, 25, 12),
            ("drone", 1, 40, 6),
            ("robot", 0.5, 10, 10),
        ]

        vehicle_id = 1
        for vehicle_type, capacity, speed, count in vehicle_configs:
            for _ in range(count):
                location = (
                    random.uniform(0, self.city_size[0]),
                    random.uniform(0, self.city_size[1]),
                )
                vehicle = Vehicle(
                    id=vehicle_id,
                    type=vehicle_type,
                    location=location,
                    capacity=capacity,
                    speed=speed,
                    battery_level=random.uniform(0.3, 1.0),
                    status="available",
                )
                self.vehicles.append(vehicle)
                vehicle_id += 1

        for sensor_id in range(847):
            location = (
                random.uniform(0, self.city_size[0]),
                random.uniform(0, self.city_size[1]),
            )
            self.traffic_sensors.append(
                TrafficSensor(
                    id=sensor_id,
                    location=location,
                    congestion_level=random.uniform(0, 0.8),
                    average_speed=random.uniform(20, 60),
                    incident_detected=random.random() < 0.05,
                    timestamp=self.current_time,
                )
            )

        for station_id in range(156):
            location = (
                random.uniform(0, self.city_size[0]),
                random.uniform(0, self.city_size[1]),
            )
            conditions = random.choice(
                ["clear", "clear", "clear", "rain", "fog"]
            )
            self.weather_stations.append(
                WeatherStation(
                    id=station_id,
                    location=location,
                    temperature=random.uniform(-5, 30),
                    precipitation=(
                        random.uniform(0, 10) if conditions == "rain" else 0
                    ),
                    wind_speed=random.uniform(0, 30),
                    visibility=(
                        random.uniform(1, 10) if conditions == "fog" else 10
                    ),
                    conditions=conditions,
                    timestamp=self.current_time,
                )
            )

        print(
            f"Initialized {len(self.vehicles)} vehicles, "
            f"{len(self.traffic_sensors)} traffic sensors, and "
            f"{len(self.weather_stations)} weather stations."
        )

    def generate_delivery_requests(self, num_requests: int = 100) -> None:
        """Generate requests and assign each to its nearest center."""
        for _ in range(num_requests):
            customer_location = (
                random.uniform(0, self.city_size[0]),
                random.uniform(0, self.city_size[1]),
            )

            # Euclidean distance is consistent with the notebook's flat map.
            mfc = min(
                self.mfc_locations,
                key=lambda location: np.sqrt(
                    (location[0] - customer_location[0]) ** 2
                    + (location[1] - customer_location[1]) ** 2
                ),
            )

            start_time = self.current_time + timedelta(
                hours=random.uniform(0, 4)
            )
            end_time = start_time + timedelta(hours=random.uniform(2, 6))
            request = DeliveryRequest(
                id=len(self.delivery_requests) + 1,
                customer_id=random.randint(1000, 9999),
                origin=mfc,
                destination=customer_location,
                priority=random.choices(
                    ["standard", "express", "urgent"],
                    weights=[0.7, 0.25, 0.05],
                )[0],
                time_window=(start_time, end_time),
                package_size=random.uniform(0.1, 2.0),
                status="pending",
                created_at=self.current_time
                + timedelta(minutes=random.uniform(0, 120)),
            )
            self.delivery_requests.append(request)

        print(f"Generated {num_requests} delivery requests.")

    def update_real_time_data(self) -> None:
        """Advance the synthetic traffic and weather observations."""
        for sensor in self.traffic_sensors:
            sensor.congestion_level = max(
                0,
                min(
                    1,
                    sensor.congestion_level + random.uniform(-0.1, 0.1),
                ),
            )
            sensor.average_speed = max(
                10,
                min(80, sensor.average_speed + random.uniform(-5, 5)),
            )
            sensor.incident_detected = random.random() < 0.02
            sensor.timestamp = self.current_time

        for station in self.weather_stations:
            station.temperature += random.uniform(-1, 1)
            if station.conditions == "rain":
                station.precipitation = max(
                    0,
                    station.precipitation + random.uniform(-2, 2),
                )
            station.wind_speed = max(
                0, station.wind_speed + random.uniform(-3, 3)
            )
            station.timestamp = self.current_time

    def predict_demand(self, hours_ahead: int = 4) -> Dict[str, float]:
        """Return a rule-based demand index for upcoming hours."""
        hourly_demand = {
            8: 1.2,
            9: 1.5,
            10: 1.3,
            11: 1.1,
            12: 0.8,
            13: 0.9,
            14: 1.0,
            15: 1.1,
            16: 1.3,
            17: 1.6,
            18: 1.4,
            19: 1.2,
            20: 0.9,
            21: 0.7,
            22: 0.5,
            23: 0.3,
        }

        if not self.weather_stations or not self.traffic_sensors:
            raise ValueError(
                "Initialize weather stations and traffic sensors before "
                "predicting demand."
            )

        rain_fraction = sum(
            station.conditions == "rain"
            for station in self.weather_stations
        ) / len(self.weather_stations)
        weather_impact = 1.0 - rain_fraction * 0.3
        average_congestion = float(
            np.mean(
                [sensor.congestion_level for sensor in self.traffic_sensors]
            )
        )
        traffic_impact = 1.0 + average_congestion * 0.2

        predictions = {}
        for hour in range(hours_ahead):
            future_hour = (self.current_time.hour + hour + 1) % 24
            base_demand = hourly_demand.get(future_hour, 1.0)
            predictions[f"hour_{hour + 1}"] = (
                base_demand * weather_impact * traffic_impact
            )

        return predictions

    def detect_disruptions(self) -> List[Dict]:
        """Apply threshold rules to identify selected system disruptions."""
        disruptions = []

        incident_sensors = [
            sensor
            for sensor in self.traffic_sensors
            if sensor.incident_detected
        ]
        if len(incident_sensors) > 3:
            disruptions.append(
                {
                    "type": "traffic_incident",
                    "severity": (
                        "high" if len(incident_sensors) > 5 else "medium"
                    ),
                    "affected_sensors": len(incident_sensors),
                    "estimated_delay": random.uniform(15, 45),
                }
            )

        rain_stations = [
            station
            for station in self.weather_stations
            if station.conditions == "rain"
        ]
        if (
            self.weather_stations
            and len(rain_stations) > len(self.weather_stations) * 0.3
        ):
            disruptions.append(
                {
                    "type": "weather_rain",
                    "severity": "medium",
                    "affected_area": len(rain_stations)
                    / len(self.weather_stations),
                    "speed_reduction": 0.25,
                }
            )

        if self.vehicles:
            available_vehicles = sum(
                vehicle.status == "available" for vehicle in self.vehicles
            )
            availability_rate = available_vehicles / len(self.vehicles)
            if availability_rate < 0.7:
                disruptions.append(
                    {
                        "type": "vehicle_shortage",
                        "severity": "high",
                        "availability_rate": availability_rate,
                        "shortage": len(self.vehicles) - available_vehicles,
                    }
                )

        return disruptions

    def run_scenario_analysis(self, disruption: Dict) -> Dict:
        """Sample 1,200 responses and return the highest-scoring candidate."""
        disruption_type = disruption["type"]
        if disruption_type not in {
            "traffic_incident",
            "weather_rain",
            "vehicle_shortage",
        }:
            raise ValueError(f"Unsupported disruption type: {disruption_type}")

        print(f"Evaluating scenarios for {disruption_type}...")
        scenarios = []

        for scenario_id in range(1200):
            if disruption_type == "traffic_incident":
                delay_reduction = random.uniform(0.1, 0.4)
                cost_increase = random.uniform(0.05, 0.15)
            elif disruption_type == "weather_rain":
                delay_reduction = random.uniform(0.05, 0.3)
                cost_increase = random.uniform(0.1, 0.25)
            else:
                delay_reduction = random.uniform(0.15, 0.35)
                cost_increase = random.uniform(0.08, 0.2)

            scenarios.append(
                {
                    "scenario_id": scenario_id,
                    "delay_reduction": delay_reduction,
                    "cost_increase": cost_increase,
                    "overall_score": delay_reduction - cost_increase * 0.5,
                }
            )

        best_scenario = max(
            scenarios, key=lambda scenario: scenario["overall_score"]
        )

        if disruption_type == "traffic_incident":
            original_delay = disruption["estimated_delay"]
            delay_savings = (
                original_delay * best_scenario["delay_reduction"]
            )
        elif disruption_type == "weather_rain":
            baseline_deliveries = 100
            affected_deliveries = int(
                baseline_deliveries * disruption["affected_area"]
            )
            mitigated_deliveries = int(
                affected_deliveries
                * (1 + best_scenario["delay_reduction"])
            )
            delivery_savings = mitigated_deliveries - affected_deliveries
            delay_savings = delivery_savings * 30
        else:
            average_delay_minutes = 60
            delay_savings = (
                average_delay_minutes
                * best_scenario["delay_reduction"]
                * disruption["shortage"]
            )

        cost_per_delay_minute = 28.40
        gross_avoided_delay_cost = (
            delay_savings * cost_per_delay_minute
        )

        return {
            "disruption_type": disruption_type,
            "scenarios_evaluated": len(scenarios),
            "best_scenario_id": best_scenario["scenario_id"],
            "delay_reduction": best_scenario["delay_reduction"],
            "cost_increase": best_scenario["cost_increase"],
            "delay_savings_minutes": delay_savings,
            "cost_savings": gross_avoided_delay_cost,
        }

    def calculate_kpis(self) -> Dict:
        """Calculate delivery, fleet, traffic, and weather indicators."""
        total_requests = len(self.delivery_requests)
        completed_requests = [
            request
            for request in self.delivery_requests
            if request.status == "delivered"
        ]
        failed_requests = [
            request
            for request in self.delivery_requests
            if request.status == "failed"
        ]

        on_time_rate = 0.0
        if completed_requests:
            on_time_deliveries = [
                request
                for request in completed_requests
                if request.estimated_delivery is not None
                and request.estimated_delivery <= request.time_window[1]
            ]
            on_time_rate = (
                len(on_time_deliveries) / len(completed_requests) * 100
            )

        vehicle_utilization = 0.0
        if self.vehicles:
            active_vehicles = sum(
                vehicle.status == "busy" for vehicle in self.vehicles
            )
            vehicle_utilization = active_vehicles / len(self.vehicles) * 100

        average_congestion = 0.0
        active_incidents = 0
        if self.traffic_sensors:
            average_congestion = float(
                np.mean(
                    [sensor.congestion_level for sensor in self.traffic_sensors]
                )
            )
            active_incidents = sum(
                sensor.incident_detected for sensor in self.traffic_sensors
            )

        weather_impact = 0.0
        if self.weather_stations:
            adverse_weather = sum(
                station.conditions in {"rain", "fog"}
                for station in self.weather_stations
            )
            weather_impact = (
                adverse_weather / len(self.weather_stations) * 100
            )

        return {
            "timestamp": self.current_time,
            "total_requests": total_requests,
            "completed_deliveries": len(completed_requests),
            "failed_deliveries": len(failed_requests),
            "on_time_rate": on_time_rate,
            "vehicle_utilization": vehicle_utilization,
            "avg_congestion": average_congestion,
            "active_incidents": active_incidents,
            "weather_impact": weather_impact,
        }


print("Digital twin classes and simulation methods are ready.")

## 2. Initialize the simulated system

Initialization creates the locations and entities used by later cells. Each delivery request is assigned to the nearest fulfillment center by straight-line distance. The fleet, sensor network, and weather stations are randomly generated, so their exact locations and conditions depend on the random draws in this run.

In [ ]:
print("=" * 60)
print("INITIALIZING THE DIGITAL TWIN SYSTEM")
print("=" * 60)

digital_twin = DigitalTwin(city_size=(100, 100))
digital_twin.initialize_system()
digital_twin.generate_delivery_requests(num_requests=150)

print("\nSYSTEM STATUS")
print(
    f"City dimensions: {digital_twin.city_size[0]} x "
    f"{digital_twin.city_size[1]} km"
)
print(f"Micro-fulfillment centers: {len(digital_twin.mfc_locations)}")
print(f"Vehicles: {len(digital_twin.vehicles)}")
print(f"Delivery requests: {len(digital_twin.delivery_requests)}")
print(
    "Synthetic traffic and weather observations: "
    f"{len(digital_twin.traffic_sensors) + len(digital_twin.weather_stations)}"
)

## 3. Update observations, estimate demand, and calculate KPIs

The update step changes the synthetic sensor observations. The demand index combines an hour-of-day baseline with two multiplicative adjustment factors:

$$D_h = B_h\,(1 - 0.3R)\,(1 + 0.2G),$$

where $D_h$ is the index for forecast hour $h$, $B_h$ is the rule-based hourly baseline, $R$ is the fraction of weather stations reporting rain, and $G$ is average sensor congestion. An index near 1 represents demand near the model's baseline. This rule is illustrative; without observed demand data, it cannot establish forecast accuracy.

The disruption detector applies explicit thresholds to incident counts, rain coverage, and vehicle availability. KPI calculations summarize the current simulated state. Since the simulation does not complete or assign deliveries, delivery-completion indicators can remain at their initial values.

In [ ]:
print("=" * 60)
print("MONITORING AND RULE-BASED DEMAND ANALYSIS")
print("=" * 60)

digital_twin.update_real_time_data()
demand_predictions = digital_twin.predict_demand(hours_ahead=4)

print("\nDemand index relative to the model baseline:")
for hour, prediction in demand_predictions.items():
    print(f"  {hour}: {prediction:.2f}x baseline")

disruptions = digital_twin.detect_disruptions()
print("\nDetected disruptions:")
if disruptions:
    for disruption in disruptions:
        print(
            f"  {disruption['type']}: "
            f"{disruption['severity']} severity"
        )
else:
    print("  None detected by the modeled threshold rules.")

current_kpis = digital_twin.calculate_kpis()
print("\nCurrent simulated system indicators:")
print(f"  Total requests: {current_kpis['total_requests']}")
print(f"  Completed deliveries: {current_kpis['completed_deliveries']}")
print(f"  Failed deliveries: {current_kpis['failed_deliveries']}")
print(f"  On-time rate: {current_kpis['on_time_rate']:.1f}%")
print(f"  Vehicle utilization: {current_kpis['vehicle_utilization']:.1f}%")
print(f"  Average congestion: {current_kpis['avg_congestion']:.2f}")
print(f"  Active sensor incidents: {current_kpis['active_incidents']}")
print(f"  Adverse-weather stations: {current_kpis['weather_impact']:.1f}%")

## 4. Evaluate response scenarios

For each detected disruption, the model samples 1,200 candidates from the ranges defined in `run_scenario_analysis`. The `overall_score` implements the scenario-selection equation above, and `max` returns the highest-scoring sampled candidate. This procedure does not guarantee a globally optimal operational response: it only selects the best candidate among the generated samples under the chosen score.

Delay savings are estimated differently by disruption type. The reported dollar amount converts those estimated minutes into gross avoided-delay cost using the assumed cost per minute. It excludes the modeled cost increase, so it should not be interpreted as net savings or as a measured real-world outcome. If no disruption is detected, a clearly labeled synthetic traffic case is evaluated to demonstrate the scenario-analysis function.

In [ ]:
print("=" * 60)
print("SCENARIO ANALYSIS")
print("=" * 60)

scenario_results = []
processing_times = []
total_cost_savings = 0.0

disruptions_to_analyze = disruptions
if not disruptions_to_analyze:
    print(
        "No disruption was detected. Evaluating a synthetic traffic "
        "incident case for demonstration."
    )
    disruptions_to_analyze = [
        {
            "type": "traffic_incident",
            "severity": "high",
            "estimated_delay": 30,
        }
    ]

for disruption in disruptions_to_analyze:
    print(f"\nAnalyzing {disruption['type']}...")
    start_time = time.perf_counter()
    result = digital_twin.run_scenario_analysis(disruption)
    processing_time = time.perf_counter() - start_time

    scenario_results.append(result)
    processing_times.append(processing_time)
    total_cost_savings += result["cost_savings"]

    print(f"  Candidates evaluated: {result['scenarios_evaluated']:,}")
    print(f"  Elapsed time: {processing_time:.4f} seconds")
    print(f"  Selected candidate ID: {result['best_scenario_id']}")
    print(f"  Estimated delay reduction: {result['delay_reduction']:.1%}")
    print(f"  Modeled cost increase fraction: {result['cost_increase']:.1%}")
    print(
        "  Estimated delay savings: "
        f"{result['delay_savings_minutes']:.1f} minutes"
    )
    print(
        "  Gross avoided-delay cost (not net savings): "
        f"${result['cost_savings']:,.2f}"
    )

actual_scenarios = sum(
    result["scenarios_evaluated"] for result in scenario_results
)
average_processing_time = float(np.mean(processing_times))
print("\nScenario-analysis summary:")
print(f"  Total candidates evaluated: {actual_scenarios:,}")
print(f"  Mean elapsed time per analysis: {average_processing_time:.4f} seconds")
print(
    "  Total estimated gross avoided-delay cost: "
    f"${total_cost_savings:,.2f}"
)

## 5. Visualize the simulated state and analysis

The dashboard combines four views: normalized system indicators, the rule-based demand index, scenario-analysis estimates, and a geographic sample of system entities. The first view is a polar chart for visual comparison, not a composite performance score. The geographic view displays only a subset of requests, incidents, and adverse-weather stations so that the map remains readable.

In [ ]:
fig = plt.figure(figsize=(15, 12))
fig.suptitle(
    "Digital Twin Simulation: Last-Mile Delivery",
    fontsize=16,
    fontweight="bold",
)
ax1 = fig.add_subplot(2, 2, 1, projection="polar")
ax2 = fig.add_subplot(2, 2, 2)
ax3 = fig.add_subplot(2, 2, 3)
ax4 = fig.add_subplot(2, 2, 4)

kpi_names = [
    "On-time rate",
    "Vehicle utilization",
    "Low congestion",
    "Low weather impact",
]
kpi_values = [
    current_kpis["on_time_rate"] / 100,
    current_kpis["vehicle_utilization"] / 100,
    1 - current_kpis["avg_congestion"],
    1 - current_kpis["weather_impact"] / 100,
]
angles = np.linspace(0, 2 * np.pi, len(kpi_names), endpoint=False)
closed_angles = np.append(angles, angles[0])
closed_values = np.append(kpi_values, kpi_values[0])
ax1.plot(closed_angles, closed_values, marker="o", linewidth=2)
ax1.fill(closed_angles, closed_values, alpha=0.2)
ax1.set_xticks(angles)
ax1.set_xticklabels(kpi_names)
ax1.set_ylim(0, 1)
ax1.set_title("Normalized system indicators", pad=20)
ax1.grid(True, alpha=0.4)

hours = list(range(1, len(demand_predictions) + 1))
predicted_demand = list(demand_predictions.values())
ax2.bar(hours, predicted_demand, color="skyblue", edgecolor="navy")
ax2.axhline(
    y=1.0,
    color="red",
    linestyle="--",
    label="Baseline index",
)
ax2.set_xlabel("Hours ahead")
ax2.set_ylabel("Relative demand index")
ax2.set_title("Rule-based demand index")
ax2.set_xticks(hours)
ax2.grid(True, axis="y", alpha=0.3)
ax2.legend()

scenario_labels = [
    result["disruption_type"].replace("_", "\n")
    for result in scenario_results
]
scenario_costs = [
    result["cost_savings"] for result in scenario_results
]
scenario_reductions = [
    result["delay_reduction"] * 100 for result in scenario_results
]
positions = np.arange(len(scenario_results))
ax3_twin = ax3.twinx()
bars_cost = ax3.bar(
    positions - 0.18,
    scenario_costs,
    width=0.36,
    color="seagreen",
    alpha=0.75,
    label="Gross avoided-delay cost ($)",
)
bars_reduction = ax3_twin.bar(
    positions + 0.18,
    scenario_reductions,
    width=0.36,
    color="darkorange",
    alpha=0.75,
    label="Estimated delay reduction (%)",
)
ax3.set_xlabel("Disruption type")
ax3.set_ylabel("Gross avoided-delay cost ($)")
ax3_twin.set_ylabel("Estimated delay reduction (%)")
ax3.set_title("Scenario estimates (not net savings)")
ax3.set_xticks(positions)
ax3.set_xticklabels(scenario_labels)
ax3.grid(True, axis="y", alpha=0.3)
ax3.legend(
    [bars_cost, bars_reduction],
    [
        "Gross avoided-delay cost ($)",
        "Estimated delay reduction (%)",
    ],
    loc="best",
    fontsize=8,
)

for index, center in enumerate(digital_twin.mfc_locations):
    ax4.scatter(
        center[0],
        center[1],
        marker="D",
        s=75,
        color="red",
        label="Fulfillment center" if index == 0 else None,
        zorder=4,
    )

request_groups = {
    "Pending request": ("pending", "o", "gold"),
    "Delivered request": ("delivered", "^", "green"),
    "Other request status": ("other", "s", "gray"),
}
for label, (status, marker, color) in request_groups.items():
    selected_requests = [
        request
        for request in digital_twin.delivery_requests[:20]
        if (
            request.status == status
            if status != "other"
            else request.status not in {"pending", "delivered"}
        )
    ]
    if selected_requests:
        ax4.scatter(
            [request.destination[0] for request in selected_requests],
            [request.destination[1] for request in selected_requests],
            marker=marker,
            color=color,
            alpha=0.75,
            label=label,
        )

incident_sensors = [
    sensor
    for sensor in digital_twin.traffic_sensors
    if sensor.incident_detected
][:10]
if incident_sensors:
    ax4.scatter(
        [sensor.location[0] for sensor in incident_sensors],
        [sensor.location[1] for sensor in incident_sensors],
        marker="X",
        color="darkorange",
        label="Traffic incident (sample)",
    )

adverse_stations = [
    station
    for station in digital_twin.weather_stations
    if station.conditions in {"rain", "fog"}
][:10]
if adverse_stations:
    ax4.scatter(
        [station.location[0] for station in adverse_stations],
        [station.location[1] for station in adverse_stations],
        marker="v",
        color="blue",
        label="Adverse-weather station (sample)",
    )

ax4.set_xlim(0, digital_twin.city_size[0])
ax4.set_ylim(0, digital_twin.city_size[1])
ax4.set_xlabel("X coordinate (km)")
ax4.set_ylabel("Y coordinate (km)")
ax4.set_title("Geographic sample of system entities")
ax4.grid(True, alpha=0.3)
ax4.legend(loc="best", fontsize=8)

fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

## 6. Interpret system indicators and scenario results

The health score below is a transparent rubric over four indicators, not a statistically validated measure of system health. It gives partial credit according to the thresholds in the code. The demand statistic describes variation among the forecast indices; it is not a measure of predictive accuracy. Accuracy would require comparing predictions with subsequently observed demand.

Recommendations are conditional prompts based on the simulated indicators. They do not constitute tested interventions, and this prototype does not implement route optimization, vehicle assignment, or a live operational control loop.

In [ ]:
print("=" * 60)
print("SIMULATED SYSTEM PERFORMANCE REVIEW")
print("=" * 60)

health_score = 0.0
maximum_score = 4.0

if current_kpis["on_time_rate"] >= 95:
    on_time_label, on_time_points = "Excellent", 1.0
elif current_kpis["on_time_rate"] >= 90:
    on_time_label, on_time_points = "Good", 0.8
elif current_kpis["on_time_rate"] >= 85:
    on_time_label, on_time_points = "Fair", 0.6
else:
    on_time_label, on_time_points = "Needs improvement", 0.0
health_score += on_time_points
print(
    f"On-time delivery rate: {current_kpis['on_time_rate']:.1f}% "
    f"({on_time_label})"
)

if 70 <= current_kpis["vehicle_utilization"] <= 90:
    utilization_label, utilization_points = "Within rubric target", 1.0
elif current_kpis["vehicle_utilization"] >= 60:
    utilization_label, utilization_points = "Moderate", 0.8
else:
    utilization_label, utilization_points = "Needs attention", 0.0
health_score += utilization_points
print(
    f"Vehicle utilization: {current_kpis['vehicle_utilization']:.1f}% "
    f"({utilization_label})"
)

if current_kpis["avg_congestion"] <= 0.3:
    congestion_label, congestion_points = "Low", 1.0
elif current_kpis["avg_congestion"] <= 0.5:
    congestion_label, congestion_points = "Moderate", 0.8
else:
    congestion_label, congestion_points = "High", 0.0
health_score += congestion_points
print(
    f"Average congestion index: {current_kpis['avg_congestion']:.2f} "
    f"({congestion_label})"
)

if current_kpis["weather_impact"] <= 20:
    weather_label, weather_points = "Low", 1.0
elif current_kpis["weather_impact"] <= 40:
    weather_label, weather_points = "Moderate", 0.8
else:
    weather_label, weather_points = "High", 0.0
health_score += weather_points
print(
    f"Stations reporting adverse weather: "
    f"{current_kpis['weather_impact']:.1f}% ({weather_label})"
)

overall_health = health_score / maximum_score * 100
print(f"\nRubric score: {overall_health:.1f}%")

forecast_variance = float(np.var(list(demand_predictions.values())))
print("\nDemand-index variability:")
print(f"  Variance across forecast hours: {forecast_variance:.3f}")
print(
    "  This statistic describes variation, not forecast accuracy; "
    "observed demand is not modeled."
)

total_scenarios = sum(
    result["scenarios_evaluated"] for result in scenario_results
)
average_delay_reduction = float(
    np.mean([result["delay_reduction"] for result in scenario_results])
)
print("\nScenario-analysis summary:")
print(f"  Candidate scenarios evaluated: {total_scenarios:,}")
print(f"  Mean selected delay-reduction fraction: {average_delay_reduction:.1%}")
print(
    "  Estimated gross avoided-delay cost: "
    f"${total_cost_savings:,.2f} (not net savings)"
)
print(
    "  Mean measured analysis runtime: "
    f"{average_processing_time:.4f} seconds per disruption"
)

print("\nConditional operational prompts:")
recommendations = []
if current_kpis["on_time_rate"] < 90:
    recommendations.append(
        "Review delivery-time assumptions and assignment decisions."
    )
if current_kpis["vehicle_utilization"] < 60:
    recommendations.append(
        "Investigate why vehicles are not recorded as busy."
    )
if current_kpis["avg_congestion"] > 0.5:
    recommendations.append(
        "Examine traffic conditions and their effect on travel times."
    )
if current_kpis["active_incidents"] > 5:
    recommendations.append(
        "Review the spatial distribution and persistence of incidents."
    )
if total_cost_savings > 0:
    recommendations.append(
        "Compare gross avoided-delay estimates with intervention costs "
        "before drawing conclusions about net benefit."
    )
if recommendations:
    for recommendation in recommendations:
        print(f"  - {recommendation}")
else:
    print("  No prompts were triggered by the selected thresholds.")

if overall_health >= 85:
    assessment = "High rubric score"
elif overall_health >= 70:
    assessment = "Moderate-to-high rubric score"
elif overall_health >= 55:
    assessment = "Intermediate rubric score"
else:
    assessment = "Low rubric score"

stream_count = (
    len(digital_twin.traffic_sensors)
    + len(digital_twin.weather_stations)
)
print(f"\nOverall rubric classification: {assessment}")
print(f"Synthetic traffic and weather observations: {stream_count:,}")
print(
    "Interpret these outputs as results of the stated simulation rules, "
    "not as live-system measurements."
)

## 7. Role and limitations of the integrated-system tier

### How this tier relates to simpler optimization models

A static optimization model takes a defined set of inputs and searches for a solution under its stated objective and constraints. An integrated digital twin is a broader system concept: it can combine operational data, state estimation, predictive models, scenario evaluation, and decision support. This notebook illustrates a few of those building blocks using synthetic data. It does not provide live synchronization, validated forecasting, or continuous operational optimization.

### Potential advantages

- Combining information from multiple operational domains can reveal dependencies that isolated models may miss.
- Scenario analysis can make assumptions about disruptions and response trade-offs explicit.
- Updated observations can support timely monitoring when the data pipeline and state-estimation process are reliable.
- A maintained history of inputs and outcomes can support later model evaluation and operational learning.

### Practical challenges

- Integrating systems requires consistent identifiers, units, timestamps, and data-quality controls.
- Forecasts and scenario scores must be validated against observed outcomes before they guide consequential decisions.
- Live infrastructure introduces reliability, latency, security, privacy, and maintenance requirements.
- A large number of data streams does not by itself establish model accuracy or decision quality.
- Scenario results depend on the candidate-generation rules, assumptions, objective weights, and constraints represented in the model.

### Appropriate use

An integrated approach is most relevant when an operation has multiple interacting data sources, meaningful uncertainty, and a clear need for coordinated decision support. Before operational deployment, the system would need domain-appropriate data, validated models, explicit feasibility constraints, monitoring, and human oversight. Those implementation steps are outside the scope of this simulation.